## 1. Environment setup and raw data loading



In [ ]:
# 01_Data_Cleaning.ipynb

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration ---
STATA_FILE_PATH = r'D:\9_Projects\Transfer learning\data\global_data_Trucost_Compustat_merged.dta'  # Replace with your original Stata file path
HDF5_FILE_PATH = os.path.join('..', 'data', 'processed_data.h5')  # Output path for the processed HDF5 file

# --- Load data ---
print('Step 1: Loading raw data from Stata (.dta)...')
try:
    df = pd.read_stata(STATA_FILE_PATH)
    print(f'Loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns.')
    print('Preview:')
    print(df.head())
    print('Data info:')
    df.info(verbose=False, memory_usage='deep')
except Exception as e:
    print(f'Failed to load data: {e}')


# 2. Target construction and validation

## 2.1 Log-transformed targets

We create log-transformed targets using \(\ln(x+1)\) to mitigate strong right-skewness in emission variables.



In [ ]:
# ==============================================================================
# (Updated) Code Block 2: Create log targets and validate via publication-quality plots (v4)
# ==============================================================================

# --- 2.1 Additional imports ---
# scipy.stats is used to fit a normal distribution and compute summary statistics.
from scipy import stats

# --- 2.2 Define and create target variables ---
ORIGINAL_TARGETS = [
    'Scope1', 'Scope2', 'Scope3_upstream',
    'Scope3_prod', 'Scope3_downLA', 'Scope_total'
]

missing_targets = [col for col in ORIGINAL_TARGETS if col not in df.columns]
if missing_targets:
    raise ValueError(f"Error: missing target columns in the dataset: {missing_targets}")

print('Step 2.1: Creating log-transformed targets (ln_*) for 6 emission variables...')
log_targets_dict = {f"ln_{col}": np.log1p(df[col]) for col in ORIGINAL_TARGETS}
log_targets_df = pd.DataFrame(log_targets_dict)
df = pd.concat([df, log_targets_df], axis=1)
LOG_TARGETS = list(log_targets_dict.keys())
print('Log-transformed columns created and merged into df.')

# --- 2.3 Publication-quality distribution comparison plots ---
print('Step 2.2: Generating the final publication-quality distribution comparison plots...')

fig, axes = plt.subplots(len(ORIGINAL_TARGETS), 2, figsize=(15, 22))
fig.suptitle('Target Variable Distribution Comparison: Original vs ln-transformed', fontsize=20, y=1.0)

for i, target in enumerate(ORIGINAL_TARGETS):
    log_target = f"ln_{target}"

    # Left: original scale
    ax_left = axes[i, 0]
    data_orig = df[target].dropna()

    # Histogram (density)
    sns.histplot(
        data_orig, ax=ax_left, stat='density', color='skyblue',
        edgecolor='white', bins=50, label='Data distribution'
    )

    # Normal fit PDF overlay
    mu_orig, std_orig = stats.norm.fit(data_orig)
    x_orig = np.linspace(data_orig.min(), data_orig.max(), 100)
    p_orig = stats.norm.pdf(x_orig, mu_orig, std_orig)
    ax_left.plot(x_orig, p_orig, 'r-', linewidth=2, label='Normal fit')

    # Summary stats box
    skew_orig = data_orig.skew()
    kurt_orig = data_orig.kurtosis()
    stats_text_orig = (
        f"Samples: {len(data_orig):,}\n"
        f"Mean: {mu_orig:.2e}\nStd: {std_orig:.2e}\n"
        f"Skewness: {skew_orig:.2f}\nKurtosis: {kurt_orig:.2f}"
    )
    ax_left.text(
        0.95, 0.95, stats_text_orig, transform=ax_left.transAxes, fontsize=9,
        va='top', ha='right', bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.5)
    )

    ax_left.set_title(f"{target} (original)", fontsize=12)
    ax_left.set_xlabel(f"{target} value")
    ax_left.set_ylabel('Probability density')
    ax_left.legend(loc='best')
    ax_left.ticklabel_format(style='sci', axis='x', scilimits=(0, 0))

    # Right: log-transformed
    ax_right = axes[i, 1]
    data_log = df[log_target].dropna()

    sns.histplot(
        data_log, ax=ax_right, stat='density', color='lightgreen',
        edgecolor='black', bins=50, label='Data distribution'
    )

    mu_log, std_log = stats.norm.fit(data_log)
    x_log = np.linspace(data_log.min(), data_log.max(), 100)
    p_log = stats.norm.pdf(x_log, mu_log, std_log)
    ax_right.plot(x_log, p_log, 'r-', linewidth=2, label='Normal fit')

    skew_log = data_log.skew()
    kurt_log = data_log.kurtosis()
    stats_text_log = (
        f"Samples: {len(data_log):,}\n"
        f"Mean: {mu_log:.3f}\nStd: {std_log:.3f}\n"
        f"Skewness: {skew_log:.2f}\nKurtosis: {kurt_log:.2f}"
Kurtosis: {kurt_log:.2f}"
    )
    ax_right.text(
        0.95, 0.95, stats_text_log, transform=ax_right.transAxes, fontsize=9,
        va='top', ha='right', bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.5)
    )

    ax_right.set_title(f"{log_target} (ln-transformed)", fontsize=12)
    ax_right.set_xlabel(f"{log_target} value")
    ax_right.set_ylabel('Probability density')
    ax_right.legend(loc='best')

fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

print('Visualization completed. Publication-quality plots with summary statistics have been generated.')

# 3. Exploratory data analysis

## 3.1 Missingness pattern analysis

We examine the missing-data distribution and patterns of carbon-emission labels across countries.



In [ ]:
# ==============================================================================
# Missingness analysis for original target variables (all countries)
# ==============================================================================

print('=' * 80)
print('Missingness analysis of original targets (all countries)')
print('=' * 80)

# --- 1) Missingness per target ---
print('1) Missingness statistics by target:')
print('-' * 50)

missing_stats = []
for target in ORIGINAL_TARGETS:
    if target in df.columns:
        total_count = len(df)
        missing_count = int(df[target].isnull().sum())
        valid_count = total_count - missing_count
        missing_pct = (missing_count / total_count) * 100 if total_count > 0 else 0

        missing_stats.append({
            'Target': target,
            'Total_Samples': total_count,
            'Valid_Count': valid_count,
            'Missing_Count': missing_count,
            'Missing_Pct': missing_pct,
        })

        print(f"{target:20s}: {valid_count:8,} valid | {missing_count:8,} missing ({missing_pct:5.1f}%)")
    else:
        print(f"{target:20s}: column not found")

missing_df = pd.DataFrame(missing_stats)

# --- 2) Missingness patterns at the row level ---
print('2) Row-level missingness patterns:')
print('-' * 50)

all_targets_missing = int(df[ORIGINAL_TARGETS].isnull().all(axis=1).sum())
all_targets_valid = int(df[ORIGINAL_TARGETS].notnull().all(axis=1).sum())
partial_missing = int(len(df) - all_targets_missing - all_targets_valid)

print(f"Rows with all targets missing: {all_targets_missing:8,} ({(all_targets_missing/len(df)*100):5.1f}%)")
print(f"Rows with all targets present: {all_targets_valid:8,} ({(all_targets_valid/len(df)*100):5.1f}%)")
print(f"Rows with partial missingness: {partial_missing:8,} ({(partial_missing/len(df)*100):5.1f}%)")

# --- 3) Country-level missingness ---
location_stats_df = None
if 'loc' in df.columns:
    print('3) Country-level missingness summary:')
    print('-' * 80)

    all_locations = df['loc'].value_counts().sort_values(ascending=False)

    print('Country | Total | All-valid | All-missing | Partial | Valid% | Usable%')
    print('-' * 80)

    location_detailed_stats = []
    for loc in all_locations.index:
        loc_data = df[df['loc'] == loc]
        total = len(loc_data)
        all_valid = int(loc_data[ORIGINAL_TARGETS].notnull().all(axis=1).sum())
        all_missing = int(loc_data[ORIGINAL_TARGETS].isnull().all(axis=1).sum())
        partial = int(total - all_valid - all_missing)

        valid_rate = (all_valid / total) * 100 if total > 0 else 0
        usable_rate = ((all_valid + partial) / total) * 100 if total > 0 else 0

        location_detailed_stats.append({
            'Location': loc,
            'Total_Samples': total,
            'All_Valid': all_valid,
            'All_Missing': all_missing,
            'Partial_Missing': partial,
            'Valid_Rate': valid_rate,
            'Usable_Rate': usable_rate,
        })

        print(f"{loc:7s} | {total:6,} | {all_valid:8,} | {all_missing:10,} | {partial:7,} | {valid_rate:6.1f} | {usable_rate:6.1f}")

    location_stats_df = pd.DataFrame(location_detailed_stats)

    print('Global summary:')
    print('-' * 50)
    print(f"# countries/regions: {len(all_locations)}")
    print(f"Top-10 by sample size: {list(all_locations.head(10).index)}")
    print(f"Mean samples per country: {all_locations.mean():.0f}")
    print(f"Median samples per country: {all_locations.median():.0f}")

    # China snapshot (if available)
    if 'CHN' in all_locations.index:
        china_stats = location_stats_df[location_stats_df['Location'] == 'CHN'].iloc[0]
        china_rank = int((all_locations >= all_locations['CHN']).sum())
        print('China (CHN) snapshot:')
        print(f"  total samples: {china_stats['Total_Samples']:,}")
        print(f"  all-valid rows: {china_stats['All_Valid']:,}")
        print(f"  partial rows: {china_stats['Partial_Missing']:,}")
        print(f"  all-missing rows: {china_stats['All_Missing']:,}")
        print(f"  completeness (all-valid): {china_stats['Valid_Rate']:.1f}%")
        print(f"  usability (>=1 label): {china_stats['Usable_Rate']:.1f}%")
        print(f"  rank by sample size: {china_rank}")
else:
    print("Warning: 'loc' column not found; country-level analysis is skipped.")

# --- 4) Year-level trend (optional) ---
yearly_df = pd.DataFrame()
if 'fiscalyear' in df.columns:
    print('4) Year-level completeness trend:')
    print('-' * 50)

    yearly_stats = []
    for year in sorted(df['fiscalyear'].dropna().unique()):
        year_data = df[df['fiscalyear'] == year]
        total = len(year_data)
        if total > 0:
            all_valid = int(year_data[ORIGINAL_TARGETS].notnull().all(axis=1).sum())
            valid_rate = (all_valid / total) * 100
            yearly_stats.append({
                'Year': int(year),
                'Total_Samples': total,
                'Complete_Records': all_valid,
                'Completeness_Rate': valid_rate,
            })

    yearly_df = pd.DataFrame(yearly_stats)
    if len(yearly_df) > 0:
        recent_years = yearly_df.tail(10)
        print('Year | Total | Complete | Complete%')
        print('-' * 40)
        for _, row in recent_years.iterrows():
            print(f"{int(row['Year'])} | {row['Total_Samples']:6,} | {row['Complete_Records']:8,} | {row['Completeness_Rate']:8.1f}%")

# --- 5) Sector-level completeness (optional) ---
sector_col = None
for col in ['GICSSector', 'gsector', 'sector']:
    if col in df.columns:
        sector_col = col
        break

if sector_col:
    print('5) Sector-level completeness (top sectors):')
    print('-' * 50)
    print('Sector | Total | All-valid | Complete%')
    print('-' * 55)

    for sector in df[sector_col].value_counts().head(15).index:
        sector_data = df[df[sector_col] == sector]
        total = len(sector_data)
        all_valid = int(sector_data[ORIGINAL_TARGETS].notnull().all(axis=1).sum())
        valid_rate = (all_valid / total) * 100 if total > 0 else 0
        sector_name = str(sector)[:20]
        print(f"{sector_name:20s} | {total:6,} | {all_valid:8,} | {valid_rate:8.1f}%")

# --- 6) Missingness correlation across targets ---
print('6) Missingness correlation among targets:')
print('-' * 50)
missing_matrix = df[ORIGINAL_TARGETS].isnull()
missing_corr = missing_matrix.corr()
print('Missingness correlation matrix (1.0 means always missing together):')
print(missing_corr.round(3))

# --- 7) Data-quality notes (high-level) ---
print('7) Data-quality notes:')     
print('-' * 50)

overall_completeness = (all_targets_valid / len(df)) * 100 if len(df) > 0 else 0
china_data = df[df['loc'] == 'CHN'] if 'loc' in df.columns else pd.DataFrame()
china_completeness = 0.0
if len(china_data) > 0:
    china_complete = int(china_data[ORIGINAL_TARGETS].notnull().all(axis=1).sum())
    china_completeness = (china_complete / len(china_data)) * 100

print(f"Overall completeness (all targets present): {overall_completeness:.1f}%")
if len(china_data) > 0:
    print(f"China completeness (all targets present): {china_completeness:.1f}%")
print(f"# complete rows usable for multi-target modeling: {all_targets_valid:,}")

if overall_completeness < 30:
    print('Low completeness: consider single-target modeling or explicit missingness handling.')
elif overall_completeness < 60:
    print('Moderate completeness: multi-target modeling is feasible, but missingness requires care.')
else:
    print('High completeness: suitable for multi-target modeling.')

print('=' * 80)


## 3.2 Descriptive statistics for the six target variables

This section reports descriptive statistics and visual summaries for the six emission targets.



In [ ]:
# ==============================================================================
# Descriptive statistics for the six target variables
# ==============================================================================

print('=' * 80)
print('Descriptive statistics for the six target variables')
print('=' * 80)

# 1) Comprehensive descriptive table
print('1) Descriptive statistics table:')
print('-' * 80)

target_stats = []
for target in ORIGINAL_TARGETS:
    if target in df.columns:
        s = df[target]
        data = s.dropna()
        if len(data) > 0:
            target_stats.append({
                'Variable': target,
                'Count': int(len(data)),
                'Missing': int(s.isnull().sum()),
                'Missing_Pct': float(s.isnull().sum() / len(df) * 100) if len(df) > 0 else 0.0,
                'Mean': float(data.mean()),
                'Std': float(data.std()),
                'Min': float(data.min()),
                'Q25': float(data.quantile(0.25)),
                'Median': float(data.median()),
                'Q75': float(data.quantile(0.75)),
                'Max': float(data.max()),
                'Skewness': float(data.skew()),
                'Kurtosis': float(data.kurtosis()),
                'CV': float(data.std() / data.mean()) if float(data.mean()) != 0 else np.nan,
                'IQR': float(data.quantile(0.75) - data.quantile(0.25)),
                'Range': float(data.max() - data.min()),
            })

stats_df = pd.DataFrame(target_stats)

# Make the table available for downstream notebooks (e.g., results analysis)
globals()['TARGET_DESCRIPTIVE_STATS'] = stats_df

# Print a compact view
if len(stats_df) > 0:
    show_cols = ['Variable', 'Count', 'Missing_Pct', 'Mean', 'Std', 'Min', 'Median', 'Max', 'Skewness', 'Kurtosis']
    print(stats_df[show_cols].to_string(index=False))
else:
    print('No target variables available for descriptive statistics.')

# 2) Correlation among targets (pairwise complete)
print('2) Target correlation (pairwise complete):') 
print('-' * 80)

corr_df = df[ORIGINAL_TARGETS].corr()
print(corr_df.round(3))


# 4. Transfer-learning candidate analysis

## 4.1 Usable sample size by country

We rank countries by the number of usable samples (NaN-aware setting) to identify candidate target domains for transfer learning.



In [ ]:
# ==============================================================================
# Country-level usable-sample analysis (sorted by usable samples)
# ==============================================================================

print('=' * 90)
print('Global country-level data completeness (sorted by usable samples)')
print('=' * 90)

if 'loc' not in df.columns:
    print("Warning: 'loc' column not found; cannot run country-level analysis.")
else:
    all_locations = df['loc'].value_counts()

    country_analysis = []
    for loc in all_locations.index:
        loc_data = df[df['loc'] == loc]
        total_samples = len(loc_data)

        all_valid = int(loc_data[ORIGINAL_TARGETS].notnull().all(axis=1).sum())
        all_missing = int(loc_data[ORIGINAL_TARGETS].isnull().all(axis=1).sum())
        partial_valid = int(total_samples - all_valid - all_missing)
        usable_samples = int(all_valid + partial_valid)  # usable = all-valid + partial-valid

        complete_rate = (all_valid / total_samples) * 100 if total_samples > 0 else 0
        usable_rate = (usable_samples / total_samples) * 100 if total_samples > 0 else 0

        country_analysis.append({
            'Country': loc,
            'Total_Samples': total_samples,
            'Usable_Samples': usable_samples,
            'Complete_Samples': all_valid,
            'Partial_Samples': partial_valid,
            'Unusable_Samples': all_missing,
            'Complete_Rate': complete_rate,
            'Usable_Rate': usable_rate,
        })

    country_df = pd.DataFrame(country_analysis)
    country_df_sorted = country_df.sort_values('Usable_Samples', ascending=True)

    print('Global overview:')
    print(f"  # countries/regions: {len(country_df)}")
    print(f"  total samples: {country_df['Total_Samples'].sum():,}")
    print(f"  usable samples (>=1 label): {country_df['Usable_Samples'].sum():,}")

    print('Transfer-learning candidate tiers (by usable samples):')
    print('-' * 90)
    print('Rank | Country | Total | Usable | Complete | Partial | Unusable | Usable% | Complete% | Tier')
    print('-' * 90)

    def tier_tag(usable: int) -> str:
        if usable < 100:
            return 'Tier-1 (very low; strong TL candidate)'
        if usable < 500:
            return 'Tier-2 (low; TL recommended)'
        if usable < 1000:
            return 'Tier-3 (medium; TL optional)'
        return 'Tier-4 (large; TL less critical)'

    for idx, (_, row) in enumerate(country_df_sorted.iterrows(), 1):
        tier = tier_tag(int(row['Usable_Samples']))
        print(
            f"{idx:4d} | {row['Country']:7s} | {int(row['Total_Samples']):6,} | {int(row['Usable_Samples']):6,} | "
            f"{int(row['Complete_Samples']):8,} | {int(row['Partial_Samples']):7,} | {int(row['Unusable_Samples']):8,} | "
            f"{row['Usable_Rate']:6.1f} | {row['Complete_Rate']:8.1f} | {tier}"
        )

    very_low = country_df_sorted[country_df_sorted['Usable_Samples'] < 100]
    low = country_df_sorted[(country_df_sorted['Usable_Samples'] >= 100) & (country_df_sorted['Usable_Samples'] < 500)]
    medium = country_df_sorted[(country_df_sorted['Usable_Samples'] >= 500) & (country_df_sorted['Usable_Samples'] < 1000)]

    # China position (if available)
    if 'CHN' in country_df['Country'].values:
        china_row = country_df[country_df['Country'] == 'CHN'].iloc[0]
        china_rank = int((country_df_sorted['Usable_Samples'] <= china_row['Usable_Samples']).sum())
        print('China (CHN) position by usable samples:')
        print(f"  rank: {china_rank} / {len(country_df)}")
        print(f"  usable samples: {int(china_row['Usable_Samples']):,}")
        print('  note: CHN typically has sufficient data and may serve as a source domain in transfer learning.')

    globals()['TRANSFER_LEARNING_CANDIDATES'] = {
        'very_low_sample': very_low['Country'].tolist(),
        'low_sample': low['Country'].tolist(),
        'medium_sample': medium['Country'].tolist(),
        'detailed_stats': country_df_sorted,
    }

    print("Saved analysis to variable 'TRANSFER_LEARNING_CANDIDATES'.")     

print('=' * 90)


# 5. Data format conversion

## 5.1 Convert DTA to HDF5

This section converts the Stata (.dta) file to an HDF5 file for faster I/O in downstream experiments.



In [ ]:
import pandas as pd
import h5py
import numpy as np
import os
import time

# --- 1) Configuration ---
dta_file_path = r'D:\9_Projects\Transfer learning\data\global_data_Trucost_Compustat_merged.dta'
hdf5_file_path = os.path.splitext(dta_file_path)[0] + '_universal_features.h5'

LABEL_COLUMN_NAMES = [
    'Scope1', 'Scope2', 'Scope3_upstream', 'Scope3_prod', 'Scope3_downLA', 'Scope_total'
]

# --- 2) Conversion logic ---
if os.path.exists(hdf5_file_path):
    print(f"HDF5 file already exists: '{hdf5_file_path}'. Skipping conversion.")
    print('To regenerate (e.g., after updating labels), please delete the file manually and rerun.')
else:
    print("Creating a 'universal' HDF5 file...")
    start_time = time.time()

    print('Reading the .dta file...')
    df = pd.read_stata(dta_file_path)
    print(f'Raw data shape: {df.shape}')

    # Check label columns
    missing_labels = [c for c in LABEL_COLUMN_NAMES if c not in df.columns]
    if missing_labels:
        raise ValueError(f"Missing label columns in input data: {missing_labels}")

    print('Splitting labels and candidate features...')
    labels_df = df[LABEL_COLUMN_NAMES].copy()
    features_df = df.drop(columns=LABEL_COLUMN_NAMES)

    # Separate numeric vs non-numeric features
    numeric_features_df = features_df.select_dtypes(include=[np.number]).copy()
    non_numeric_df = features_df.select_dtypes(exclude=[np.number]).copy()
    non_numeric_names = list(non_numeric_df.columns)
    all_feature_names = list(numeric_features_df.columns)

    print(f'Numeric features (model inputs): {len(all_feature_names)}')
    if non_numeric_names:
        print(
            f"Non-numeric columns retained (not model inputs; saved for metadata): {len(non_numeric_names)}. "
            f"First 10: {non_numeric_names[:10]}"
        )

    # Labels: keep NaN; clip negative values to 0 then apply log1p
    print('Processing original and log-transformed labels...')
    labels_original_np = labels_df.to_numpy(dtype=np.float32)

    neg_mask = ~np.isnan(labels_original_np)
    if np.any((labels_original_np < 0) & neg_mask):
        print('Found negative labels; clipping to 0 before log1p.')
        labels_clipped = np.where(
            np.isnan(labels_original_np),
            np.nan,
            np.clip(labels_original_np, a_min=0.0, a_max=None),
        )
    else:
        labels_clipped = labels_original_np

    labels_log_np = np.log1p(labels_clipped)

    # Feature matrix (numeric only)
    features_matrix = numeric_features_df.to_numpy(dtype=np.float32)

    # Non-numeric matrix stored as UTF-8 strings
    if non_numeric_names:
        non_numeric_matrix = non_numeric_df.astype(str).to_numpy()
    else:
        non_numeric_matrix = np.empty((len(df), 0), dtype='U1')

    print(f'Feature matrix shape: {features_matrix.shape}')
    print(f'Non-numeric matrix shape: {non_numeric_matrix.shape}')
    print(f'Original labels shape: {labels_original_np.shape}')
    print(f'Log labels shape: {labels_log_np.shape}')

    print('Writing to HDF5...')
    string_dt = h5py.string_dtype(encoding='utf-8')
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('features', data=features_matrix, compression='gzip')
        hf.create_dataset('feature_names', data=np.array(all_feature_names, dtype=object), dtype=string_dt)

        hf.create_dataset('non_numeric_feature_names', data=np.array(non_numeric_names, dtype=object), dtype=string_dt)
        hf.create_dataset('non_numeric_data', data=non_numeric_matrix, dtype=string_dt, compression='gzip')

        hf.create_dataset('labels_log', data=labels_log_np, compression='gzip')
        hf.create_dataset('labels_original', data=labels_original_np, compression='gzip')

    end_time = time.time()
    print('Conversion completed.')
    print(f'Elapsed time: {end_time - start_time:.2f} seconds')
    print(f'New universal HDF5 file saved to: {hdf5_file_path}')
